In [1]:
%load_ext autoreload
%autoreload 2

import os 
import sys

import scipy
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder

import pandas as pd
import numpy as np

from statsmodels.discrete.conditional_models import ConditionalLogit

from preproces_prod3 import *
    
local_path = os.getcwd()
code_root = os.path.abspath(os.path.join(local_path, '..', 'Code'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)
code_root = os.path.abspath(os.path.join(local_path, '../..', 'inv'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)

from matching_case_control import call_data, comparar_medias_test,analyze_vrs_data, integer_programming_matching_gurobi, match_nn_max_dist_weigths, charly_mip, charly_double_mip

from IPython.core.display import display
warnings.filterwarnings("ignore")

In [2]:

display.max_output_lines = 500  # Adjust the number as needed
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'

In [5]:
icd10_codes_fr = [
    "N390",   # Infección del tracto urinario (sin síntomas respiratorios) INFECCION DE VÍAS URINARIAS, SITIO NO ESPECIFICADO
    "A090",     # Gastroenteritis aguda (sin síntomas respiratorios) OTRAS GASTROENTERITIS Y COLITIS NO ESPECIFICADAS DE ORIGEN INFECCIOSO
    "A099",     # Gastroenteritis aguda (sin síntomas respiratorios) GASTROENTERITIS Y COLITIS DE ORIGEN NO ESPECIFICADO
    "A09X",     # Gastroenteritis aguda (sin síntomas respiratorios) DIARREA Y GASTROENTERITIS DE PRESUNO ORIGEN INFECCIOSO
    "R100",  # Cólico infantil (sin fiebre ni síntomas respiratorios) ABDOMEN AGUDO
    "R101",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR ABDOMINAL LOCALIZADO EN PARTE SUPERIOR
    "R102",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR PELVICO Y PERINEAL
    "R103",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR LOCALIZADO EN OTRAS PARTES INFERIORES DEL ABDOMEN
    "R104",  # Cólico infantil (sin fiebre ni síntomas respiratorios) OTROS DOLORES ABDOMINALES Y LOS NO ESPECIFICADOS
    "R11X",  # Cólico infantil (sin fiebre ni síntomas respiratorios) NAUSEA Y VOMITO
    "R634",   # Pérdida de peso (sin fiebre ni síntomas respiratorios) PERDIDA ANORMAL DE PESO
    "R633",   # Dificultades de alimentación (sin fiebre ni síntomas respiratorios) DIFICULTADES Y MALA ADMINISTRACION DE LA ALIMENTACION
    "P599",   # Ictericia neonatal (sin fiebre ni síntomas respiratorios) ICTERICIA NEONATAL, NO ESPECIFICADA
    "R681",  # Llanto anormal (sin fiebre ni síntomas respiratorios) SINTOMAS NO ESPECIFICOS PROPIOS DE LA INFANCIA
    "S099",  # Traumatismo craneal (sin fiebre ni síntomas respiratorios)
    "Z539"    # Cirugía de emergencia (sin fiebre ni síntomas respiratorios)
]

In [6]:
df_all_vrs, df_all_upc = call_data('NAC_RNI_EGRESOS_ENTREGA_ISCI_20_12_2024_encr.csv')
df_vrs = (
    df_all_vrs
    #.query('(FECHA_ING > "2024-03-31") | FECHA_ING.isna()')
    .copy()
)

df_upc = (
    df_all_upc
    #.query('(FECHA_ING > "2024-03-31") | FECHA_ING.isna()')
    .copy()
)

df_vrs_match_case = df_vrs.copy()
df_vrs_match_case.to_csv(path_data/'df_vrs_match_case_s39_2012.csv') #, index=False

df_upc_match_case = df_upc.copy()
df_upc_match_case.to_csv(path_data/'df_upc_match_case_s39_2012.csv') #, index=False

n_rows_inicial= 167609


c:\Users\ntrig\Desktop\ISCI\Proyectos\Efectividad_Nirse\Code\preproces_prod3.py:194: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df[cols].applymap(is_in_range).any(axis=1)
c:\Users\ntrig\Desktop\ISCI\Proyectos\Efectividad_Nirse\Code\preproces_prod3.py:194: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df[cols].applymap(is_in_range).any(axis=1)


['DESCONOCIDO---->None' 'Región De Antofagasta---->ANTOFAGASTA'
 'Región De Arica Parinacota---->ARICA Y PARINACOTA'
 'Región De Atacama---->ATACAMA'
 'Región De Aysén del General Carlos Ibañez del Campo---->AISEN'
 'Región De Coquimbo---->COQUIMBO' 'Región De La Araucanía---->ARAUCANIA'
 'Región De Los Lagos---->LOS LAGOS' 'Región De Los Ríos---->LOS RIOS'
 'Región De Magallanes y de la Antártica Chilena---->MAGALLANES Y ANTARTICA'
 'Región De Tarapacá---->TARAPACA' 'Región De Valparaíso---->VALPARAISO'
 'Región De Ñuble---->NUBLE' 'Región Del Bíobío---->BIOBIO'
 "Región Del Libertador Gral. B. O'Higgins---->O'HIGGINS"
 'Región Del Maule---->MAULE'
 'Región Metropolitana de Santiago---->METROPOLITANA']
n_rows_post_prefiltred= 167609
Datos perdidos por muertes:  1202
ruts perdidos por filtro semanas y peso:  645
Droped intersex: 9
Datos perdidos por edad madre atípica: 245
Datos perdidos por fecha ingreso menor a fecha nacimiento: 57
vrs en los primeros 7 dias de nacer: 5
Ruts eliminad

c:\Users\ntrig\Desktop\ISCI\Proyectos\Efectividad_Nirse\Code\preproces_prod3.py:482: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_f[nombre_columna] = df_f['fechaInm'] + pd.DateOffset(days=d)
c:\Users\ntrig\Desktop\ISCI\Proyectos\Efectividad_Nirse\Code\preproces_prod3.py:484: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_f[nombre_columna_bin] = 1 - df_f[nombre_columna].isna()
c:\Users\ntrig\Desktop\ISCI\Proyectos\Efectividad_Nirse\Code\preproces_prod3.py:482: PerformanceWarning: DataFrame is highly fragmented.  This is usu

In [7]:
np.random.seed(42)
dict_experiments={}

## Todos

### general

In [ ]:
df = df_vrs_match_case.copy()
df_case= df.query("diag_vrs==True").copy()
df_control= df.query("diag_1_leter!='J'").copy()  # or diag_1_leter.isna()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

N° de obs case 1275 y control 149827


In [19]:
#Primer experimento: hacemos matching x ESTABLECIMEINTO
# Paso 1: Aplicar el matching con opciones de intervalos
intervals = {'PESO': 300}  # Por ejemplo, 5 kilos y 3 días de intervalo
match_vars = ['SEXO','PESO','NOMBRE_REGION','COMUNA','group'] #[,'COMUNA_N' 'ESTAB'
match_date_vars =  {'fecha_nac': 28} 
# Aplicamos la función de matching con la variable de fecha
matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)


df_matched = matched_data.copy()
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()
display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEXO','SEMANAS','edad_relativa']))

# Llamada a la función
results = analyze_vrs_data(df_matched)

# Imprimir resultados relevantes
print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

creacion conjuntos A_i time: 303.73056173324585
creacion variables time: 0.2942068576812744
optimize model time: 0.05256247520446777
matched_data time: 0.6973447799682617
Total cases matched is : 1223
Dado tu estado de inmunizacion cual es probabilidad de estar contagiado


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,3227.35,3218.78,0.37,0.71,
1,SEXO,1.38,1.38,0.00,1.00,
2,SEMANAS,38.22,37.85,4.62,0.00,Existe una diferencia significativa.
3,edad_relativa,157.95,139.77,5.42,0.00,Existe una diferencia significativa.


Tabla de Contingencia:
 Estado de Inmunización    0     1   All
Diagnóstico VRS                        
0                        88  1135  1223
1                       318   905  1223
All                     406  2040  2446

Tabla de Porcentajes:
 Estado de Inmunización           0           1    All
Diagnóstico VRS                                      
0                        21.674877   55.637255   50.0
1                        78.325123   44.362745   50.0
All                     100.000000  100.000000  100.0

Odds Ratio:  0.22
Efectividad:  77.9 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                 2446
Model:               ConditionalLogit   No. groups:                       1223
Log-Likelihood:               -763.73   Min group size:                      2
Method:                          BFGS   Max group size:                      2
Date:                Mon,

In [20]:
weights = {'edad_relativa': 1} #,'ingreso_relativo':2 

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa'], 
                                                              match_vars_exact = ['SEXO','SEMANAS','group','COMUNA','NOMBRE_REGION'],
                                                              match_vars_onehot=[],
                                                              ratio="1:2",
                                                              max_distance=1,
                                                              weights=weights)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

Total cases = 1275, Total controls = 149827
Total cases matched is : 1189, Total control matched is : 2378
ratio: 1:2
No matched : 86


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,3256.23,3265.70,-0.51,0.61,
1,SEMANAS,38.06,38.06,0.00,1.00,
2,SEXO,1.37,1.37,0.00,1.00,
3,edad_relativa,141.12,140.50,0.21,0.83,




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.715909
1    0.279501
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización    0     1   All
Diagnóstico VRS                        
0                       125  2253  2378
1                       315   874  1189
All                     440  3127  3567

Tabla de Porcentajes:
 Estado de Inmunización           0           1         All
Diagnóstico VRS                                           
0                        28.409091   72.049888   66.666667
1                        71.590909   27.950112   33.333333
All                     100.000000  100.000000  100.000000

Odds Ratio:  0.15
Efectividad:  84.6 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                 3567
Model:               ConditionalLogit   No. groups:                       1189
Log

### upc

In [21]:
df = df_upc_match_case.copy()
df_case= df.query("diag_vrs==True").copy()
df_control= df.query("diag_vrs==False").copy()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

N° de obs case 268 y control 153498


In [23]:
intervals = {'PESO': 300,'SEMANAS':1}  # Por ejemplo, 5 kilos y 3 días de intervalo
match_vars = ['SEXO','PESO','NOMBRE_REGION','COMUNA','group','muy_prematuro','prematuro','SEMANAS'] #[,'COMUNA_N' 'ESTAB'
match_date_vars =  {'fecha_nac': 14} 
# Aplicamos la función de matching con la variable de fecha
matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)


df_matched = matched_data.copy()
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()
display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','edad_relativa']))

# Llamada a la función
results = analyze_vrs_data(df_matched)

# Imprimir resultados relevantes
print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

creacion conjuntos A_i time: 63.26010346412659
creacion variables time: 0.010035514831542969
optimize model time: 0.003019094467163086
matched_data time: 0.14165616035461426
Total cases matched is : 239
Dado tu estado de inmunizacion cual es probabilidad de estar contagiado


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,3190.62,3192.50,-0.04,0.97,
1,SEMANAS,37.93,37.88,0.28,0.78,
2,edad_relativa,147.93,140.40,0.99,0.32,


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                       12  227  239
1                       76  163  239
All                     88  390  478

Tabla de Porcentajes:
 Estado de Inmunización           0           1    All
Diagnóstico VRS                                      
0                        13.636364   58.205128   50.0
1                        86.363636   41.794872   50.0
All                     100.000000  100.000000  100.0

Odds Ratio:  0.11
Efectividad:  88.7 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  478
Model:               ConditionalLogit   No. groups:                        239
Log-Likelihood:               -132.67   Min group size:                      2
Method:                          BFGS   Max group size:                      2
Date:                Mon, 10 Feb 2025   

In [24]:
weights = {'edad_relativa': 1,'PESO': 0.4} #,'ingreso_relativo':2 

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa','PESO'], 
                                                              match_vars_exact = ['SEXO','SEMANAS','group','COMUNA','NOMBRE_REGION','muy_prematuro','prematuro'],
                                                              match_vars_onehot=[],
                                                              ratio="1:2",
                                                              max_distance=5,
                                                              weights=weights)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

Total cases = 268, Total controls = 153498
Total cases matched is : 253, Total control matched is : 506
ratio: 1:2
No matched : 15


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,3191.73,3187.25,0.10,0.92,
1,SEMANAS,37.88,37.88,0.00,1.00,
2,SEXO,1.36,1.36,0.00,1.00,
3,edad_relativa,142.53,141.13,0.22,0.83,




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.785714
1    0.266263
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                       21  485  506
1                       77  176  253
All                     98  661  759

Tabla de Porcentajes:
 Estado de Inmunización           0           1         All
Diagnóstico VRS                                           
0                        21.428571   73.373676   66.666667
1                        78.571429   26.626324   33.333333
All                     100.000000  100.000000  100.000000

Odds Ratio:  0.099
Efectividad:  90.1 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  759
Model:               ConditionalLogit   No. groups:                        253
Log-Likelihood:  

## Francia

### general

In [ ]:
df = df_vrs_match_case.copy()
df_case= df.query("diag_vrs==True").copy()
df_control= df[(df.DIAG1.isin(icd10_codes_fr)) & (df.fechaIng_any.notna())].copy()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

N° de obs case 1275 y control 1451


In [36]:
intervals = {'PESO': 3000} 
match_vars = ['SEXO','PESO','NOMBRE_REGION','group','SEMANAS'] 
match_date_vars =  {'fecha_nac': 20, 'fechaIng_any':28} 

matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)


df_matched = matched_data.copy()
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()
display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','edad_relativa','ingreso_relativo']))

# Llamada a la función
results = analyze_vrs_data(df_matched)

# Imprimir resultados relevantes
print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

creacion conjuntos A_i time: 6.980454206466675
creacion variables time: 0.001996755599975586
optimize model time: 0.0010006427764892578
matched_data time: 0.45267748832702637
Total cases matched is : 458
Dado tu estado de inmunizacion cual es probabilidad de estar contagiado


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,3356.43,3322.04,1.15,0.25,
1,SEMANAS,38.33,38.33,0.00,1.00,
2,edad_relativa,157.97,153.90,0.72,0.47,
3,ingreso_relativo,102.07,104.66,-0.82,0.41,


Tabla de Contingencia:
 Estado de Inmunización    0    1  All
Diagnóstico VRS                      
0                        13  445  458
1                       108  350  458
All                     121  795  916

Tabla de Porcentajes:
 Estado de Inmunización           0           1    All
Diagnóstico VRS                                      
0                        10.743802   55.974843   50.0
1                        89.256198   44.025157   50.0
All                     100.000000  100.000000  100.0

Odds Ratio:  0.095
Efectividad:  90.5 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  916
Model:               ConditionalLogit   No. groups:                        458
Log-Likelihood:               -271.72   Min group size:                      2
Method:                          BFGS   Max group size:                      2
Date:                Mon, 10 Feb 2

In [39]:
weights = {'edad_relativa': 3,'ingreso_relativo':3}

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa','ingreso_relativo'], 
                                                              match_vars_exact = ['sexo','SEMANAS','group','region'],
                                                              match_vars_onehot=[],
                                                              ratio="1:1",
                                                              max_distance=6,
                                                              weights=weights)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','edad_relativa','ingreso_relativo']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

Total cases = 1275, Total controls = 1451
Total cases matched is : 661, Total control matched is : 661
ratio: 1:1
No matched : 614


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,3309.87,3281.82,1.03,0.30,
1,SEMANAS,38.07,38.07,0.00,1.00,
2,edad_relativa,155.16,145.49,2.03,0.04,Existe una diferencia significativa.
3,ingreso_relativo,83.81,95.09,-3.77,0.00,Existe una diferencia significativa.




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.866953
1    0.421488
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización    0     1   All
Diagnóstico VRS                        
0                        31   630   661
1                       202   459   661
All                     233  1089  1322

Tabla de Porcentajes:
 Estado de Inmunización           0          1    All
Diagnóstico VRS                                     
0                        13.304721   57.85124   50.0
1                        86.695279   42.14876   50.0
All                     100.000000  100.00000  100.0

Odds Ratio:  0.11
Efectividad:  88.8 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                 1322
Model:               ConditionalLogit   No. groups:                        661
Log-Likelihood:               -37

### upc

In [44]:
df = df_upc_match_case.copy()
df_case= df.query("diag_vrs==True").copy()
df_control= df[(df.DIAG1.isin(icd10_codes_fr)) & (df.fechaIng_any.notna())].copy()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

N° de obs case 268 y control 1457


In [45]:
intervals = {'PESO': 3000,'SEMANAS':1} 
match_vars = ['SEXO','PESO','NOMBRE_REGION','group','SEMANAS','prematuro','muy_prematuro'] 
match_date_vars =  {'fecha_nac': 28, 'fechaIng_any':28} 

matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)


df_matched = matched_data.copy()
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()
display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','edad_relativa','ingreso_relativo']))

# Llamada a la función
results = analyze_vrs_data(df_matched)

# Imprimir resultados relevantes
print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

creacion conjuntos A_i time: 1.6080608367919922
creacion variables time: 0.0019996166229248047
optimize model time: 0.0010006427764892578
matched_data time: 0.08869433403015137
Total cases matched is : 177
Dado tu estado de inmunizacion cual es probabilidad de estar contagiado


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,3307.37,3226.51,1.44,0.15,
1,SEMANAS,37.99,38.12,-0.79,0.43,
2,edad_relativa,154.38,144.62,1.09,0.27,
3,ingreso_relativo,106.52,108.48,-0.45,0.65,


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                        7  170  177
1                       53  124  177
All                     60  294  354

Tabla de Porcentajes:
 Estado de Inmunización           0           1    All
Diagnóstico VRS                                      
0                        11.666667   57.823129   50.0
1                        88.333333   42.176871   50.0
All                     100.000000  100.000000  100.0

Odds Ratio:  0.096
Efectividad:  90.4 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  354
Model:               ConditionalLogit   No. groups:                        177
Log-Likelihood:               -99.516   Min group size:                      2
Method:                          BFGS   Max group size:                      2
Date:                Mon, 10 Feb 2025  

In [47]:
weights = {'edad_relativa': 3,'ingreso_relativo':4,'SEMANAS':5}

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa','ingreso_relativo','SEMANAS'], 
                                                              match_vars_exact = ['SEXO','group','NOMBRE_REGION','muy_prematuro','prematuro'],
                                                              match_vars_onehot=[],
                                                              ratio="1:1",
                                                              max_distance=5,
                                                              weights=weights)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa','ingreso_relativo']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

Total cases = 268, Total controls = 1457
Total cases matched is : 219, Total control matched is : 219
ratio: 1:1
No matched : 49


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,3280.15,3194.00,1.68,0.09,
1,SEMANAS,38.00,38.00,0.06,0.95,
2,SEXO,1.36,1.36,0.00,1.00,
3,edad_relativa,151.97,146.76,0.63,0.53,
4,ingreso_relativo,106.44,108.63,-0.53,0.60,




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.864198
1    0.417367
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                       11  208  219
1                       70  149  219
All                     81  357  438

Tabla de Porcentajes:
 Estado de Inmunización           0           1    All
Diagnóstico VRS                                      
0                        13.580247   58.263305   50.0
1                        86.419753   41.736695   50.0
All                     100.000000  100.000000  100.0

Odds Ratio:  0.11
Efectividad:  88.7 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  438
Model:               ConditionalLogit   No. groups:                        219
Log-Likelihood:               -123.15   Min

# prematuros

## Todos

### general

In [13]:
### general
df = df_vrs_match_case.copy().query('32<=SEMANAS<=36')
df_case= df.query("diag_vrs==True").copy()
df_control= df.query("diag_vrs==False").copy()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

N° de obs case 177 y control 13148


In [18]:
intervals = {'PESO': 300}  
match_vars = ['SEXO','PESO','NOMBRE_REGION','group','SEMANAS'] #[,'COMUNA_N' 'ESTAB'
match_date_vars =  {'fecha_nac': 28} 

matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)


df_matched = matched_data.copy()
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()
display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEXO','SEMANAS','edad_relativa']))

# Llamada a la función
results = analyze_vrs_data(df_matched)

# Imprimir resultados relevantes
print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

KeyboardInterrupt: 

In [23]:

distance_vars = ['PESO','fecha_nac','SEMANAS'] #[,'COMUNA_N' 'ESTAB'

matched_data, feasible_controls = charly_mip(df_cases=df_case,df_control=df_control, distance_vars=distance_vars, exact_var_match = ['sexo','region','group'], ratio="1:3")


display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

creacion conjuntos A_i time: 4.480377435684204
creacion variables time: 214.62710762023926
optimize model time: 0.49376702308654785
matched_data time: 0.9410109519958496
Total cases matched is : 531


c:\Users\ntrig\Desktop\ISCI\Proyectos\Efectividad_Nirse\Code\matching_case_control.py:692: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matched_data['Group'] = matched_data['Matched_Case_RUN'].fillna(matched_data['RUN'])


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,2505.50,2503.81,0.04,0.97,
1,SEMANAS,34.84,34.82,0.17,0.87,
2,SEXO,1.36,1.36,0.00,1.00,
3,edad_relativa,141.30,140.21,0.15,0.88,




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.600000
1    0.223404
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                       20  511  531
1                       30  147  177
All                     50  658  708

Tabla de Porcentajes:
 Estado de Inmunización      0           1    All
Diagnóstico VRS                                 
0                        40.0   77.659574   75.0
1                        60.0   22.340426   25.0
All                     100.0  100.000000  100.0

Odds Ratio:  0.19
Efectividad:  80.8 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  708
Model:               ConditionalLogit   No. groups:                        177
Log-Likelihood:               -229.58   Min group size:             

In [27]:
distance_vars = ['PESO','fecha_nac','SEMANAS'] #[,'COMUNA_N' 'ESTAB'

matched_data, feasible_controls = charly_double_mip(df_cases=df_case,df_control=df_control, distance_vars=distance_vars, exact_var_match = ['sexo','region','group'], ratio="1:3")


display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

creacion conjuntos A_i time: 4.434087038040161
creacion variables time: 247.17308235168457
optimize model 1 time: 0.7165451049804688
optimize model 2 time: 1.0470008850097656
matched_data time: 0.9329943656921387
Total cases = 177, Total controls = 13148
Total cases matched is : 531, Total control matched is : 531
ratio: 1:3


c:\Users\ntrig\Desktop\ISCI\Proyectos\Efectividad_Nirse\Code\matching_case_control.py:805: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,2505.50,2503.81,0.04,0.97,
1,SEMANAS,34.84,34.82,0.17,0.87,
2,SEXO,1.36,1.36,0.00,1.00,
3,edad_relativa,141.30,140.21,0.15,0.88,




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.600000
1    0.223404
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                       20  511  531
1                       30  147  177
All                     50  658  708

Tabla de Porcentajes:
 Estado de Inmunización      0           1    All
Diagnóstico VRS                                 
0                        40.0   77.659574   75.0
1                        60.0   22.340426   25.0
All                     100.0  100.000000  100.0

Odds Ratio:  0.19
Efectividad:  80.8 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  708
Model:               ConditionalLogit   No. groups:                        177
Log-Likelihood:               -229.58   Min group size:             

In [28]:
weights = {'edad_relativa': 1} #,'ingreso_relativo':2 

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa'], 
                                                              match_vars_exact = ['SEXO','SEMANAS','group','NOMBRE_REGION'],
                                                              match_vars_onehot=[],
                                                              ratio="1:3",
                                                              max_distance=1,
                                                              weights=weights)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

c:\Users\ntrig\Desktop\ISCI\Proyectos\venv\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
c:\Users\ntrig\Desktop\ISCI\Proyectos\venv\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
c:\Users\ntrig\Desktop\ISCI\Proyectos\venv\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
c:\Users\ntrig\Desktop\ISCI\Proyectos\venv\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(
c:\Users\ntrig\Desktop\ISCI\Proyectos\venv\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but NearestNeighbors was fitted with feature names
  warnings.warn(


Total cases = 177, Total controls = 13148
Total cases matched is : 176, Total control matched is : 528
ratio: 1:3
No matched : 1


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,2540.30,2510.48,0.67,0.5,
1,SEMANAS,34.84,34.84,0.00,1.0,
2,SEXO,1.36,1.36,0.00,1.0,
3,edad_relativa,140.09,140.12,-0.00,1.0,




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.652174
1    0.221884
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                       16  512  528
1                       30  146  176
All                     46  658  704

Tabla de Porcentajes:
 Estado de Inmunización           0          1    All
Diagnóstico VRS                                     
0                        34.782609   77.81155   75.0
1                        65.217391   22.18845   25.0
All                     100.000000  100.00000  100.0

Odds Ratio:  0.15
Efectividad:  84.8 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  704
Model:               ConditionalLogit   No. groups:                        176
Log-Likelihood:               -224.86   Min grou

### upc

In [33]:
df = df_upc_match_case.copy().query('32<=SEMANAS<=36')
df_case= df.query("diag_vrs==True").copy()
df_control= df.query("diag_vrs==False").copy()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

N° de obs case 45 y control 13280


In [55]:
intervals = {'PESO': 300,'SEMANAS':1}  # Por ejemplo, 5 kilos y 3 días de intervalo
match_vars = ['SEXO','PESO','NOMBRE_REGION','COMUNA','group','muy_prematuro','prematuro','SEMANAS'] #[,'COMUNA_N' 'ESTAB'
match_date_vars =  {'fecha_nac': 14} 
# Aplicamos la función de matching con la variable de fecha
matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)


df_matched = matched_data.copy()
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()
display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','edad_relativa']))

# Llamada a la función
results = analyze_vrs_data(df_matched)

# Imprimir resultados relevantes
print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

creacion conjuntos A_i time: 1.2940926551818848
creacion variables time: 0.0010001659393310547
optimize model time: 0.0
matched_data time: 0.016449451446533203
Total cases matched is : 31
Dado tu estado de inmunizacion cual es probabilidad de estar contagiado


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,2412.97,2457.10,-0.36,0.72,
1,SEMANAS,35.06,34.94,0.44,0.66,
2,edad_relativa,139.87,138.94,0.05,0.96,


Tabla de Contingencia:
 Estado de Inmunización  0   1  All
Diagnóstico VRS                   
0                       0  31   31
1                       7  24   31
All                     7  55   62

Tabla de Porcentajes:
 Estado de Inmunización      0           1    All
Diagnóstico VRS                                 
0                         0.0   56.363636   50.0
1                       100.0   43.636364   50.0
All                     100.0  100.000000  100.0

Odds Ratio:  0.0
Efectividad:  1e+02 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                   62
Model:               ConditionalLogit   No. groups:                         31
Log-Likelihood:               -16.636   Min group size:                      2
Method:                          BFGS   Max group size:                      2
Date:                Mon, 10 Feb 2025   Mean group size:                   

In [34]:
weights = {'edad_relativa': 1,'PESO': 0.4} #,'ingreso_relativo':2 

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa','PESO'], 
                                                              match_vars_exact = ['SEXO','SEMANAS','group','COMUNA','NOMBRE_REGION','muy_prematuro','prematuro'],
                                                              match_vars_onehot=[],
                                                              ratio="1:3",
                                                              max_distance=5,
                                                              weights=weights)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

Total cases = 45, Total controls = 13280
Total cases matched is : 36, Total control matched is : 108
ratio: 1:3
No matched : 9


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,2529.35,2470.00,0.66,0.51,
1,SEMANAS,34.97,34.97,0.00,1.00,
2,SEXO,1.36,1.36,0.00,1.00,
3,edad_relativa,153.03,142.94,0.60,0.55,




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.545455
1    0.225564
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                        5  103  108
1                        6   30   36
All                     11  133  144

Tabla de Porcentajes:
 Estado de Inmunización           0           1    All
Diagnóstico VRS                                      
0                        45.454545   77.443609   75.0
1                        54.545455   22.556391   25.0
All                     100.000000  100.000000  100.0

Odds Ratio:  0.24
Efectividad:  75.7 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  144
Model:               ConditionalLogit   No. groups:                         36
Log-Likelihood:               -47.541   Min

In [35]:
distance_vars = ['PESO','fecha_nac','SEMANAS'] #[,'COMUNA_N' 'ESTAB'

matched_data, feasible_controls = charly_double_mip(df_cases=df_case,df_control=df_control, distance_vars=distance_vars, exact_var_match = ['sexo','region','group'], ratio="1:3")


display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

creacion conjuntos A_i time: 1.2553720474243164
creacion variables time: 55.0181999206543
optimize model 1 time: 0.03699898719787598
optimize model 2 time: 0.18000173568725586
matched_data time: 0.15399789810180664
Total cases = 45, Total controls = 13280
Total cases matched is : 135, Total control matched is : 45
ratio: 1:3


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,2416.38,2421.22,-0.06,0.95,
1,SEMANAS,34.85,34.82,0.14,0.89,
2,SEXO,1.36,1.36,0.00,1.00,
3,edad_relativa,139.64,138.20,0.10,0.92,




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.733333
1    0.206061
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                        4  131  135
1                       11   34   45
All                     15  165  180

Tabla de Porcentajes:
 Estado de Inmunización           0           1    All
Diagnóstico VRS                                      
0                        26.666667   79.393939   75.0
1                        73.333333   20.606061   25.0
All                     100.000000  100.000000  100.0

Odds Ratio:  0.094
Efectividad:  90.6 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  180
Model:               ConditionalLogit   No. groups:                         45
Log-Likelihood:               -53.298   Mi

## Francia

### general

In [73]:
df = df_vrs_match_case.copy().query('32<=SEMANAS<=36')
df_case= df.query("diag_vrs==True").copy()
df_control= df[(df.DIAG1.isin(icd10_codes_fr))].copy()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

N° de obs case 177 y control 834


In [63]:
intervals = {'PESO': 3000,'SEMANAS':1} 
match_vars = ['SEXO','PESO','NOMBRE_REGION','group','SEMANAS'] 
match_date_vars =  {'fecha_nac': 36, 'fechaIng_any':36} 

matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)


df_matched = matched_data.copy()
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()
display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','edad_relativa','ingreso_relativo']))

# Llamada a la función
results = analyze_vrs_data(df_matched)

# Imprimir resultados relevantes
print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

creacion conjuntos A_i time: 0.6390907764434814
creacion variables time: 0.0
optimize model time: 0.0009984970092773438
matched_data time: 0.038001298904418945
Total cases matched is : 65
Dado tu estado de inmunizacion cual es probabilidad de estar contagiado


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,2545.23,2570.86,-0.35,0.73,
1,SEMANAS,35.32,35.22,0.70,0.49,
2,edad_relativa,168.89,163.29,0.38,0.71,
3,ingreso_relativo,104.94,110.88,-0.86,0.39,


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                        2   63   65
1                       10   55   65
All                     12  118  130

Tabla de Porcentajes:
 Estado de Inmunización           0           1    All
Diagnóstico VRS                                      
0                        16.666667   53.389831   50.0
1                        83.333333   46.610169   50.0
All                     100.000000  100.000000  100.0

Odds Ratio:  0.17
Efectividad:  82.5 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  130
Model:               ConditionalLogit   No. groups:                         65
Log-Likelihood:               -39.510   Min group size:                      2
Method:                          BFGS   Max group size:                      2
Date:                Mon, 10 Feb 2025   

In [74]:
weights = {'edad_relativa': 3} #,'ingreso_relativo':3

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa'], #,'ingreso_relativo'
                                                              match_vars_exact = ['sexo','SEMANAS','group','region'],
                                                              match_vars_onehot=[],
                                                              ratio="1:2",
                                                              max_distance=3,
                                                              weights=weights)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','edad_relativa','ingreso_relativo']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

Total cases = 177, Total controls = 834
Total cases matched is : 124, Total control matched is : 248
ratio: 1:2
No matched : 53


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,2719.08,2611.10,2.13,0.03,Existe una diferencia significativa.
1,SEMANAS,35.26,35.26,0.00,1.00,
2,edad_relativa,144.58,142.55,0.21,0.83,
3,ingreso_relativo,24.73,112.55,-8.88,0.00,Existe una diferencia significativa.




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.740741
1    0.301449
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                        7  241  248
1                       20  104  124
All                     27  345  372

Tabla de Porcentajes:
 Estado de Inmunización           0           1         All
Diagnóstico VRS                                           
0                        25.925926   69.855072   66.666667
1                        74.074074   30.144928   33.333333
All                     100.000000  100.000000  100.000000

Odds Ratio:  0.15
Efectividad:  84.9 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  372
Model:               ConditionalLogit   No. groups:                        124
Log-Likelihood:   

### upc

In [39]:
df = df_upc_match_case.copy().query('32<=SEMANAS<=36')
df_case= df.query("diag_vrs==True").copy()
df_control= df[df.DIAG1.isin(icd10_codes_fr)].copy()
print(f"N° de obs case {df_case.shape[0]} y control {df_control.shape[0]}")

N° de obs case 45 y control 840


In [40]:
intervals = {'PESO': 3000,'SEMANAS':1} 
match_vars = ['SEXO','PESO','NOMBRE_REGION','group','SEMANAS','prematuro','muy_prematuro'] 
match_date_vars =  {'fecha_nac': 28, 'fechaIng_any':28} 

matched_data, feasible_controls = integer_programming_matching_gurobi(df_cases=df_case,
                                 df_control=df_control,
                                 match_vars=match_vars,
                                 match_date_vars=match_date_vars,
                                 intervals=intervals)


df_matched = matched_data.copy()
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean()
display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','edad_relativa','ingreso_relativo']))

# Llamada a la función
results = analyze_vrs_data(df_matched)

# Imprimir resultados relevantes
print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

creacion conjuntos A_i time: 0.21509623527526855
creacion variables time: 0.0
optimize model time: 0.0
matched_data time: 0.009026765823364258
Total cases matched is : 17
Dado tu estado de inmunizacion cual es probabilidad de estar contagiado


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,2494.71,2354.12,1.14,0.26,
1,SEMANAS,34.94,35.18,-0.77,0.45,
2,edad_relativa,172.29,163.47,0.34,0.74,
3,ingreso_relativo,96.88,97.47,-0.04,0.97,


Tabla de Contingencia:
 Estado de Inmunización  0   1  All
Diagnóstico VRS                   
0                       0  17   17
1                       5  12   17
All                     5  29   34

Tabla de Porcentajes:
 Estado de Inmunización      0          1    All
Diagnóstico VRS                                
0                         0.0   58.62069   50.0
1                       100.0   41.37931   50.0
All                     100.0  100.00000  100.0

Odds Ratio:  0.0
Efectividad:  1e+02 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                   34
Model:               ConditionalLogit   No. groups:                         17
Log-Likelihood:               -8.3180   Min group size:                      2
Method:                          BFGS   Max group size:                      2
Date:                Tue, 11 Feb 2025   Mean group size:                   2.0
T

In [41]:
weights = {'edad_relativa': 3} #,'ingreso_relativo':4

matched_data, matched_incompleto, matched_controls = match_nn_max_dist_weigths(df_control, df_case,
                                                              match_vars_nn=['edad_relativa'], #, 'ingreso_relativo'
                                                              match_vars_exact = ['SEXO','group','NOMBRE_REGION','prematuro'],
                                                              match_vars_onehot=[],
                                                              ratio="1:2",
                                                              max_distance=5,
                                                              weights=weights)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa','ingreso_relativo']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

Total cases = 45, Total controls = 840
Total cases matched is : 42, Total control matched is : 84
ratio: 1:2
No matched : 3


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,2636.67,2398.45,2.65,0.01,Existe una diferencia significativa.
1,SEMANAS,35.01,34.74,1.24,0.22,
2,SEXO,1.36,1.36,0.00,1.00,
3,edad_relativa,140.89,135.52,0.35,0.72,
4,ingreso_relativo,19.56,109.74,-5.55,0.00,Existe una diferencia significativa.




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.818182
1    0.286957
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                        2   82   84
1                        9   33   42
All                     11  115  126

Tabla de Porcentajes:
 Estado de Inmunización           0           1         All
Diagnóstico VRS                                           
0                        18.181818   71.304348   66.666667
1                        81.818182   28.695652   33.333333
All                     100.000000  100.000000  100.000000

Odds Ratio:  0.089
Efectividad:  91.1 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  126
Model:               ConditionalLogit   No. groups:                         42
Log-Likelihood:  

In [42]:
distance_vars = ['PESO','fecha_nac','SEMANAS'] #[,'COMUNA_N' 'ESTAB'

matched_data, feasible_controls = charly_double_mip(df_cases=df_case,df_control=df_control, distance_vars=distance_vars, exact_var_match = ['sexo','region','group'], ratio="1:3")

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

creacion conjuntos A_i time: 0.1050560474395752
creacion variables time: 2.8003716468811035
optimize model 1 time: 0.004003763198852539
optimize model 2 time: 0.005985736846923828
matched_data time: 0.0850827693939209
Total cases = 45, Total controls = 840
Total cases matched is : 43, Total control matched is : 129
ratio: 1:3


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,PESO,2487.94,2418.95,0.90,0.37,
1,SEMANAS,34.93,34.77,0.81,0.42,
2,SEXO,1.37,1.37,0.00,1.00,
3,edad_relativa,136.57,138.81,-0.15,0.88,




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.818182
1    0.211180
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización   0    1  All
Diagnóstico VRS                     
0                        2  127  129
1                        9   34   43
All                     11  161  172

Tabla de Porcentajes:
 Estado de Inmunización           0           1    All
Diagnóstico VRS                                      
0                        18.181818   78.881988   75.0
1                        81.818182   21.118012   25.0
All                     100.000000  100.000000  100.0

Odds Ratio:  0.059
Efectividad:  94.1 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                  172
Model:               ConditionalLogit   No. groups:                         43
Log-Likelihood:               -50.719   Mi

In [53]:
df_vrs_match_case.query('is_rural==1')

,RUN,RUN_RNI,RUN_MADRE,VACUNADO,MARCA,FECHA_NACIMIENTO,MES_NAC,ANO_NAC,SEXO,SEMANAS,PESO,TALLA,EDAD_M,INS_C_M,INS_N_M,COMUNA,COMUNA_N,REG_RES,URBA_RURAL,NAC_MA,FECHA_INMUNIZACION,FECHA_DEFUNCION,CAUSA_DEFUNCION,VIVO,FALLECIDO_PREVIO,ESTAB,ServicioSalud,Seremi,P_ORIGEN,PREVI,FECHA_INGRESO,FECHA_EGRESO,AREA_FUNC_I,SER_CLIN_I,DIA_1_TRAS,MES_1_TRAS,ANO_1_TRAS,AREAF_1_TRAS,SERC_1_TRAS,DIA_2_TRAS,MES_2_TRAS,ANO_2_TRAS,AREAF_2_TRAS,SERC_2_TRAS,DIA_3_TRAS,MES_3_TRAS,ANO_3_TRAS,AREAF_3_TRAS,SERC_3_TRAS,DIA_4_TRAS,MES_4_TRAS,ANO_4_TRAS,AREAF_4_TRAS,SERC_4_TRAS,DIA_5_TRAS,MES_5_TRAS,ANO_5_TRAS,AREAF_5_TRAS,SERC_5_TRAS,DIA_6_TRAS,MES_6_TRAS,ANO_6_TRAS,AREAF_6_TRAS,SERC_6_TRAS,DIA_7_TRAS,MES_7_TRAS,ANO_7_TRAS,AREAF_7_TRAS,SERC_7_TRAS,DIA_8_TRAS,MES_8_TRAS,ANO_8_TRAS,AREAF_8_TRAS,SERC_8_TRAS,DIA_9_TRAS,MES_9_TRAS,ANO_9_TRAS,AREAF_9_TRAS,SERC_9_TRAS,AREAF_EGR,DIAS_ESTAD,COND_EGR,DIAG1,DIAG2,DIAG3,DIAG4,DIAG5,DIAG6,DIAG7,DIAG8,DIAG9,DIAG10,DIAG11,NOMBRE_REGION,Porcentaje Urbano,porcent_rural,percent_poor_multidim,percent_poor,p_00001_lognormal,p_99999_lognormal,fecha_tras_1,fecha_tras_2,fecha_tras_3,fecha_tras_4,fecha_tras_5,fecha_tras_6,fecha_tras_7,fecha_tras_8,fecha_tras_9,fecha_nac,fechaIng_any,fechaEgr,fechaInm,VRS_D1,VRS_D1y3,VRS_Dall,diag_irag,diag_ira_alta,LRTI_Flag,LRTI_all_j,LRTI_Flag_Dall,fechaIng_vrs,fechaIng_LRTI,fechaIng_vrs_Dall,fechaIng_LRTI_Dall,group,sexo,prematuro_extremo,muy_prematuro,prematuro_moderado,prematuro,atypic_mom_age,region,Macrozona2,cama,fecha_upc,fecha_upc_vrs,fecha_upc_vrs_Dall,days_upc,dias_en_ing,days_estad_vrs,days_estad_vrs_Dall,is_rural,categori_macro,categori_regions,exp_rural,is_poor,vrs_pre_campaña,lrti_pre_campaña,any_pre_campaña,upc_pre_campaña,dias_en_area_1,dias_en_area_2,dias_en_area_3,dias_en_area_4,dias_en_area_5,dias_en_area_6,dias_en_area_7,dias_en_area_8,dias_en_area_9,days_upc_vrs,days_upc_vrs_Dall,event_upc,event_upc_Dall,event,event_vrs_Dall,event_LRTI,event_any,take_nirse,fechaIng_vrs_copy,age_1m,si_1_meses,age_2m,si_2_meses,age_3m,si_3_meses,age_4m,si_4_meses,age_5m,si_5_meses,age_6m,si_6_meses,inm_7d,inm_mayor_7d,inm_14d,inm_mayor_14d,inm_21d,inm_mayor_21d,inm_28d,inm_mayor_28d,inm_35d,inm_mayor_35d,inm_42d,inm_mayor_42d,inm_49d,inm_mayor_49d,inm_56d,inm_mayor_56d,inm_63d,inm_mayor_63d,inm_70d,inm_mayor_70d,inm_77d,inm_mayor_77d,inm_84d,inm_mayor_84d,inm_91d,inm_mayor_91d,inm_98d,inm_mayor_98d,inm_105d,inm_mayor_105d,inm_112d,inm_mayor_112d,inm_119d,inm_mayor_119d,inm_126d,inm_mayor_126d,inm_133d,inm_mayor_133d,inm_140d,inm_mayor_140d,inm_147d,inm_mayor_147d,inm_154d,inm_mayor_154d,inm_161d,inm_mayor_161d,inm_168d,inm_mayor_168d,inm_175d,inm_mayor_175d,inm_182d,inm_mayor_182d,inmunizado,chile_chico,estado_inmunizacion,diag_vrs,diag_1_leter,edad_relativa,ingreso_relativo
21,10253b42b8d3d30da94f230355b7441910063bdbefdd23e626038eea08589056,1,1,SI,0,06NOV2023,11,2023,2,38.0,2892.0,47.0,28,2,1,16104,16104,16.0,0,C,2024-04-05,NaN,NaN,SI,VIVO,117101.0,17.0,NaN,152.0,1.0,09APR2024,11APR2024,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,0,2.0,1.0,J219,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Región De Ñuble,0.421497,0.578503,0.189575,14.924781,1873.189390,5271.639121,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,2023-11-06,2024-04-09,2024-04-11,NaT,1,1,1,True,False,True,True,True,2024-04-09,2024-04-09,2024-04-09,2024-04-09,CATCH_UP,0,0,0,0,0,0,NUBLE,Centro,,NaT,NaT,NaT,0,0,2.0,2.0,1,1,13,1.783367,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,True,2024-04-09,2023-12-06,1,2024-01-06,1,2024-02-06,1,2024-03-06,1,2024-04-06,1,2024-05-06,1,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,NaT,0,0,0,0,1,J,9,8.0
57,dadfaaea5c57aa6d61fae3d3b87c5bec85e09d87b4a17251b063ccca6a0e7577,1,1,SI,0,07FEB2024,2,2024,1,31.0,1645.0,40.0,31,8,4,9205,9205,9.0,0,C,2024-06-06,NaN,NaN,SI,VIVO,129108.0,29.0,NaN,152.